# Experiment 1: Physical Inverse-Distance Acoustic Law Verification ($1/r$ and $1/r^2$)
This notebook performs an experimental verification of **spherical acoustic wave decay** using the PYNQ-Z2 hardware overlay (`v1.6.0`).

### Physical Theory
In a 3D open acoustic field, sound emitted by a point source propagates as a spherical wave:
- **Acoustic Pressure Amplitude:** $p(r) \propto \frac{1}{r} \implies V_{\text{RMS}}(r) \propto r^{-1.0}$
- **Acoustic Intensity / Energy:** $I(r) \propto p^2(r) \propto \frac{1}{r^2} \implies I(r) \propto r^{-2.0}$

We measure $V_{\text{RMS}}(r)$ across calibrated physical distances $r \in [0.25\,\text{m}, 2.00\,\text{m}]$ across multiple frequencies ($500\,\text{Hz}$, $1000\,\text{Hz}$, $2500\,\text{Hz}$) through the hardware spectral mask and IFFT engine.

## 1. Initialize Hardware Overlay

In [ ]:
import time
import numpy as np
import plotly.graph_objects as go
from pynq_localizer import MicrophoneArrayOverlay, KinematicAnalytics

# Load overlay in Full Audio profile (50 kSPS, N=1024)
ol = MicrophoneArrayOverlay()
ol.set_profile("audio", packet_size=2048, fft_len=1024)

print(f"✅ Overlay Loaded: {ol.current_profile} profile ({ol.fs_per_ch:.0f} SPS/ch, N={ol.fft_len})")
print(f"   Filter initialized in: {ol.filter}")

## 2. Interactive Distance Sweep Data Acquisition
Place a smartphone or speaker emitting a pure tone at calibrated physical distances from the PYNQ-Z2 microphones.

For each distance $r$, click **Run** (or press Enter when prompted) to capture the live physical RMS voltage through the hardware IFFT engine.

In [ ]:
# Calibration distances in meters
test_distances_m = [0.25, 0.50, 0.75, 1.00, 1.50, 2.00]
test_freq_hz = 1000.0  # Test tone frequency
band_half_hz = 100.0   # Bandpass half-bandwidth

# Configure hardware bandpass filter around the test tone
ol.filter.set_bandpass(center_hz=test_freq_hz, delta_hz=band_half_hz)
print(f"Hardware Filter Configured: {ol.filter}")

measured_voltages_rms = []

print("\n--- STARTING PHYSICAL DISTANCE SWEEP ---")
for r in test_distances_m:
    input(f"👉 Place speaker at exactly r = {r:.2f} m and start {test_freq_hz:.0f} Hz tone. Press [ENTER] to capture...")
    
    # Capture real physical signals
    v_raw_a0, v_raw_a1, v_filt, freqs, mags = ol.capture_all()
    
    # Compute AC RMS of the hardware filtered IFFT signal
    v_filt_ac = v_filt - np.mean(v_filt)
    v_rms = float(np.sqrt(np.mean(v_filt_ac ** 2)))
    measured_voltages_rms.append(v_rms)
    
    print(f"   ✅ Captured r = {r:.2f} m -> Filtered V_RMS = {v_rms:.4f} V")

ol.filter.bypass()
print("\n✅ Sweep Complete!")

## 3. Physical Curve Fitting & Statistical Validation ($1/r$ Law)
We perform linear regression on the logarithmic transform:
$$\ln(V_{\text{RMS}}) = -n \ln(r) + \ln(A)$$
and check if the measured exponent $n \approx 1.00$ (indicating $I \propto r^{-2}$). 

In [ ]:
distances_arr = np.array(test_distances_m)
voltages_arr = np.array(measured_voltages_rms)

# Perform physical log-log regression
results = KinematicAnalytics.fit_inverse_distance_law(distances_arr, voltages_arr)

print("===========================================================")
print("  📊 INVERSE-DISTANCE ACOUSTIC REGRESSION RESULTS")
print("===========================================================")
print(f"  • Measured Voltage Exponent (n)  : {results['measured_exponent_n']:.3f}  (Theoretical Ideal = 1.000)")
print(f"  • Measured Intensity Exponent    : {results['intensity_exponent_2n']:.3f}  (Theoretical Ideal = 2.000)")
print(f"  • Goodness of Fit (R²)           : {results['r_squared']:.4f}")
print(f"  • Deviation from Ideal 1/r Law   : {results['error_pct_from_ideal_1_over_r']:.2f}%")
print("===========================================================")

# Dense distance axis for plotting theoretical fit
r_dense = np.linspace(min(distances_arr) * 0.9, max(distances_arr) * 1.1, 200)
v_fitted = results['amplitude_coefficient_A'] * (r_dense ** (-results['measured_exponent_n']))

# Plot 1: Linear Scale (Physical Decay Curve)
fig_lin = go.Figure()
fig_lin.add_trace(go.Scatter(x=distances_arr, y=voltages_arr, mode='markers+text', text=[f"{v:.3f}V" for v in voltages_arr], textposition="top right", name='Measured V_RMS', marker=dict(size=10, color='#FFA500')))
fig_lin.add_trace(go.Scatter(x=r_dense, y=v_fitted, mode='lines', name=f"Fit: V ~ r^(-{results['measured_exponent_n']:.2f})", line=dict(color='#00FFCC', width=2)))
fig_lin.update_layout(template='plotly_dark', title='<b>Acoustic Distance Decay: Measured V_RMS vs. Distance r</b>', xaxis_title='Distance r (meters)', yaxis_title='RMS Voltage (Volts)', height=420)
fig_lin.show()

# Plot 2: Log-Log Scale (Slope Verification: Ideal Slope = -1.0)
fig_log = go.Figure()
fig_log.add_trace(go.Scatter(x=np.log(distances_arr), y=np.log(voltages_arr), mode='markers', name='ln(V) Measured', marker=dict(size=10, color='#FFA500')))
fig_log.add_trace(go.Scatter(x=np.log(r_dense), y=np.log(v_fitted), mode='lines', name=f"Slope = -{results['measured_exponent_n']:.3f}", line=dict(color='#00FFCC', width=2)))
fig_log.update_layout(template='plotly_dark', title='<b>Log-Log Regression: ln(V_RMS) vs. ln(r) (Ideal Slope = -1.000)</b>', xaxis_title='ln(r)', yaxis_title='ln(V_RMS)', height=420)
fig_log.show()

## 4. Multi-Frequency Transfer Linearity Check
Test that the IFFT reconstruction preserves amplitude identically across different frequencies ($500\,\text{Hz}, 1000\,\text{Hz}, 2500\,\text{Hz}$) at a fixed distance.

In [ ]:
test_freqs = [500.0, 1000.0, 2500.0]
fixed_distance = 0.50  # Fixed distance in meters

print(f"Place speaker at fixed distance r = {fixed_distance} m.")
for f in test_freqs:
    input(f"👉 Play {f:.0f} Hz tone at {fixed_distance}m and press [ENTER]...")
    ol.filter.set_bandpass(center_hz=f, delta_hz=100.0)
    v_raw_a0, _, v_filt, _, _ = ol.capture_all()
    rms_raw = np.sqrt(np.mean((v_raw_a0 - np.mean(v_raw_a0))**2))
    rms_filt = np.sqrt(np.mean((v_filt - np.mean(v_filt))**2))
    ratio = rms_filt / max(rms_raw, 1e-6)
    print(f"   f = {f:4.0f} Hz | Raw RMS = {rms_raw:.4f}V | Filtered RMS = {rms_filt:.4f}V | Transfer Ratio = {ratio:.3f}")

ol.filter.bypass()
ol.close()
print("\n🔒 Experiment 1 Complete. Hardware cleanly released.")